# Steering Codon Usage with SAE Features

This notebook demonstrates **feature steering** on the CodonFM Encodon 1B model.
We use a trained sparse autoencoder (SAE) to identify interpretable features in the
model's residual stream, then intervene on those features at inference time to
control codon usage predictions.

**The demo:**
1. Take a real primate gene sequence
2. Mask codon positions and let the model re-predict them
3. Steer SAE features to shift predictions toward human-optimized or de-optimized codons
4. Measure the effect via Codon Adaptation Index (CAI) and GC3 content

In [ ]:
import sys
import json
from pathlib import Path
from collections import Counter

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# -- Path setup --
RECIPE_DIR = Path(".").resolve().parent
REPO_ROOT = RECIPE_DIR.parent.parent.parent.parent
CODONFM_TE_DIR = REPO_ROOT / "recipes" / "codonfm_ptl_te"
sys.path.insert(0, str(CODONFM_TE_DIR))
sys.path.insert(0, str(RECIPE_DIR / "src"))

from src.inference.encodon import EncodonInference
from src.data.preprocess.codon_sequence import process_item
from src.tokenizer import Tokenizer

from sae.architectures import TopKSAE
from sae.steering import SteeredModel, Intervention, InterventionMode

## 1. Configuration

Set paths to the Encodon 1B checkpoint and the trained SAE checkpoint.

In [ ]:
# -- Paths (edit these) --
MODEL_PATH = ""       # e.g. "/data/codonfm/checkpoints/NV-CodonFM-Encodon-TE-1B-v1"
SAE_CHECKPOINT = ""   # e.g. "../outputs/merged_1b/checkpoints/checkpoint_final.pt"
CSV_PATH = ""         # e.g. "/path/to/Primates.csv"

LAYER = -2            # penultimate layer (where SAE was trained)
CONTEXT_LENGTH = 2048
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

print(f"Device: {DEVICE}")

## 2. Load Model & SAE

In [ ]:
# Load Encodon 1B
inference = EncodonInference(model_path=MODEL_PATH, task_type="embedding_prediction", use_transformer_engine=True)
inference.configure_model()
inference.model.to(DEVICE).eval()

tokenizer = inference.tokenizer
encodon = inference.model.model  # the inner EnCodon nn.Module

num_layers = len(encodon.layers)
target_layer = LAYER if LAYER >= 0 else num_layers + LAYER
print(f"Encodon: {num_layers} layers, hooking layer {target_layer}")
print(f"Hidden dim: {encodon.config.hidden_size}")

In [ ]:
# Load SAE
def load_sae(checkpoint_path, top_k_override=None):
    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    state_dict = ckpt["model_state_dict"]
    if any(k.startswith("module.") for k in state_dict):
        state_dict = {k.removeprefix("module."): v for k, v in state_dict.items()}
    w = state_dict["encoder.weight"]
    hidden_dim, input_dim = w.shape
    model_config = ckpt.get("model_config", {})
    top_k = top_k_override or model_config.get("top_k", 32)
    sae = TopKSAE(
        input_dim=input_dim, hidden_dim=hidden_dim, top_k=top_k,
        normalize_input=model_config.get("normalize_input", False),
    )
    sae.load_state_dict(state_dict)
    return sae

sae = load_sae(SAE_CHECKPOINT).eval().to(DEVICE)
print(f"SAE: {sae.input_dim} -> {sae.hidden_dim} features (top-k={sae.top_k})")

## 3. Set Up Steering

The `SteeredModel` registers a forward hook on the target transformer layer.
When interventions are active, it intercepts the residual stream, encodes through
the SAE, modifies feature activations, and decodes back.

In [ ]:
# SteeredModel expects to find .layers on the model
# EnCodon has .layers directly, so this works with the generic resolver
steered = SteeredModel(encodon, sae, layer=target_layer, device=DEVICE)
print(f"Steering hook registered on layer {target_layer}")

## 4. Load a Test Sequence

In [ ]:
import csv

# Read sequences from Primates.csv
with open(CSV_PATH) as f:
    reader = csv.DictReader(f)
    rows = list(reader)

# Pick a medium-length gene (200-400 codons)
seq_col = "cds" if "cds" in rows[0] else "seq"
candidates = [(i, r) for i, r in enumerate(rows) if 600 <= len(r[seq_col]) <= 1200]
row = candidates[0][1] if candidates else rows[0]
dna_seq = row[seq_col]

# Split into codons
codons = [dna_seq[i:i+3] for i in range(0, len(dna_seq) - len(dna_seq) % 3, 3)]
n_codons = len(codons)
print(f"Sequence: {row.get('seq_id', 'unknown')}")
print(f"Length: {n_codons} codons ({n_codons * 3} nt)")
print(f"First 20 codons: {' '.join(codons[:20])}")

In [ ]:
# Codon-to-amino-acid table
CODON_TO_AA = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
    'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
    'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G',
}

# Human codon usage frequencies (per 1000 codons, Kazusa)
HUMAN_CODON_USAGE = {
    'TTT': 17.6, 'TTC': 20.3, 'TTA': 7.7, 'TTG': 12.9,
    'CTT': 13.2, 'CTC': 19.6, 'CTA': 7.2, 'CTG': 39.6,
    'ATT': 16.0, 'ATC': 20.8, 'ATA': 7.5, 'ATG': 22.0,
    'GTT': 11.0, 'GTC': 14.5, 'GTA': 7.1, 'GTG': 28.1,
    'TCT': 15.2, 'TCC': 17.7, 'TCA': 12.2, 'TCG': 4.4,
    'CCT': 17.5, 'CCC': 19.8, 'CCA': 16.9, 'CCG': 6.9,
    'ACT': 13.1, 'ACC': 18.9, 'ACA': 15.1, 'ACG': 6.1,
    'GCT': 18.4, 'GCC': 27.7, 'GCA': 15.8, 'GCG': 7.4,
    'TAT': 12.2, 'TAC': 15.3, 'TAA': 1.0, 'TAG': 0.8,
    'CAT': 10.9, 'CAC': 15.1, 'CAA': 12.3, 'CAG': 34.2,
    'AAT': 17.0, 'AAC': 19.1, 'AAA': 24.4, 'AAG': 31.9,
    'GAT': 21.8, 'GAC': 25.1, 'GAA': 29.0, 'GAG': 39.6,
    'TGT': 10.6, 'TGC': 12.6, 'TGA': 1.6, 'TGG': 13.2,
    'CGT': 4.5, 'CGC': 10.4, 'CGA': 6.2, 'CGG': 11.4,
    'AGT': 12.1, 'AGC': 19.5, 'AGA': 12.2, 'AGG': 12.0,
    'GGT': 10.8, 'GGC': 22.2, 'GGA': 16.5, 'GGG': 16.5,
}

# Amino acid translation
aa_seq = "".join(CODON_TO_AA.get(c, "?") for c in codons)
print(f"Protein: {aa_seq[:60]}...")

## 5. Metrics: CAI and GC3

- **CAI** (Codon Adaptation Index): measures how well codon usage matches the host organism's preferences. Higher = more optimized for human expression.
- **GC3**: fraction of G or C at the wobble (3rd) position. Human-preferred codons tend to have high GC3.

In [ ]:
def compute_relative_adaptiveness():
    """Compute w_ij = freq(codon) / max_freq(synonymous codons) for CAI."""
    # Group codons by amino acid
    aa_groups = {}
    for codon, aa in CODON_TO_AA.items():
        if aa == '*':  # skip stop codons
            continue
        aa_groups.setdefault(aa, []).append(codon)
    
    w = {}
    for aa, group_codons in aa_groups.items():
        max_freq = max(HUMAN_CODON_USAGE[c] for c in group_codons)
        for c in group_codons:
            w[c] = HUMAN_CODON_USAGE[c] / max_freq
    return w

W_CAI = compute_relative_adaptiveness()


def compute_cai(codon_list):
    """Compute CAI for a list of codons (excluding stop codons)."""
    coding = [c for c in codon_list if CODON_TO_AA.get(c, '*') != '*']
    if not coding:
        return 0.0
    log_sum = sum(np.log(W_CAI.get(c, 0.01)) for c in coding)
    return np.exp(log_sum / len(coding))


def compute_gc3(codon_list):
    """Fraction of G/C at the wobble (3rd) position."""
    coding = [c for c in codon_list if len(c) == 3 and CODON_TO_AA.get(c, '*') != '*']
    if not coding:
        return 0.0
    gc3 = sum(1 for c in coding if c[2] in ('G', 'C'))
    return gc3 / len(coding)


print(f"Original sequence:")
print(f"  CAI  = {compute_cai(codons):.4f}")
print(f"  GC3  = {compute_gc3(codons):.4f}")

## 6. Masked Prediction Helper

We mask a subset of codon positions, run the model forward, and decode the
top-1 predictions back to codons.

In [ ]:
MASK_ID = tokenizer.mask_token_id  # 4


def predict_masked(dna_sequence, mask_positions, model_pl=inference.model):
    """
    Mask specified codon positions, run Encodon forward, return predicted codons.
    
    Args:
        dna_sequence: raw DNA string
        mask_positions: list of 0-indexed codon positions to mask
        model_pl: the EncodonPL model (hooks fire on inner EnCodon automatically)
    
    Returns:
        dict mapping codon_position -> predicted_codon_string
    """
    item = process_item(dna_sequence, context_length=CONTEXT_LENGTH, tokenizer=tokenizer)
    input_ids = torch.tensor(item["input_ids"], dtype=torch.long).unsqueeze(0).to(DEVICE)
    attn_mask = torch.tensor(item["attention_mask"], dtype=torch.long).unsqueeze(0).to(DEVICE)
    
    # Mask positions (offset by 1 for CLS token)
    for pos in mask_positions:
        input_ids[0, pos + 1] = MASK_ID
    
    with torch.no_grad():
        output = model_pl({"input_ids": input_ids, "attention_mask": attn_mask})
    
    logits = output.logits[0]  # [seq_len, vocab_size]
    predictions = {}
    for pos in mask_positions:
        token_idx = pos + 1  # offset for CLS
        pred_id = logits[token_idx].argmax().item()
        pred_codon = tokenizer.decoder.get(pred_id, "???")
        predictions[pos] = pred_codon
    
    return predictions


# Quick test: mask first 5 positions
test_preds = predict_masked(dna_seq, list(range(5)))
for pos, pred in test_preds.items():
    orig = codons[pos]
    aa_match = CODON_TO_AA.get(pred, '?') == CODON_TO_AA.get(orig, '!')
    print(f"  pos {pos}: {orig} -> {pred}  (AA: {CODON_TO_AA.get(orig,'?')} -> {CODON_TO_AA.get(pred,'?')})  {'ok' if aa_match else 'AA CHANGED'}")

## 7. Define Steering Features

From the feature analysis, we select features associated with codon usage bias:

- **"common codons | wobble GC"** features push toward human-preferred codons
- **"wobble AT"** features push toward rare codons with A/T at the 3rd position

Feature IDs come from `feature_labels.json`.

In [ ]:
# Load feature labels if available
labels_path = RECIPE_DIR / "codon_dashboard" / "dist" / "feature_labels.json"
if labels_path.exists():
    with open(labels_path) as f:
        feature_labels = json.load(f)
    print(f"Loaded {len(feature_labels)} feature labels")
else:
    feature_labels = {}
    print("No feature labels found; using hardcoded feature IDs")

In [ ]:
# Auto-discover features by label pattern, or use hardcoded fallbacks
def find_features(labels, pattern):
    """Find feature IDs whose labels contain the given pattern."""
    return [int(k) for k, v in labels.items() if pattern in v.lower()]


if feature_labels:
    gc_features = find_features(feature_labels, "wobble gc")
    at_features = find_features(feature_labels, "wobble at")
    print(f"Found {len(gc_features)} 'wobble GC' features")
    print(f"Found {len(at_features)} 'wobble AT' features")
else:
    # Hardcoded from earlier exploration
    gc_features = [86, 151, 160, 166, 191, 206, 211, 216, 221]
    at_features = [21, 62, 341, 640]

# Use a manageable subset for steering
GC_STEER_FEATURES = gc_features[:10]
AT_STEER_FEATURES = at_features[:5]

print(f"\nSteering with:")
print(f"  GC features ({len(GC_STEER_FEATURES)}): {GC_STEER_FEATURES}")
for fid in GC_STEER_FEATURES:
    print(f"    {fid}: {feature_labels.get(str(fid), '?')}")
print(f"  AT features ({len(AT_STEER_FEATURES)}): {AT_STEER_FEATURES}")
for fid in AT_STEER_FEATURES:
    print(f"    {fid}: {feature_labels.get(str(fid), '?')}")

## 8. Run Steering Experiment

Three conditions:
1. **Baseline** -- no steering, natural model predictions
2. **Human-optimized** -- amplify wobble-GC features
3. **De-optimized** -- amplify wobble-AT features

In [ ]:
# Mask every 3rd codon (keep enough context for the model)
mask_positions = list(range(1, n_codons - 1, 3))  # skip first (ATG) and last
print(f"Masking {len(mask_positions)} / {n_codons} positions ({100*len(mask_positions)/n_codons:.0f}%)")

In [ ]:
STEER_WEIGHT = 15.0  # steering strength (tune this)


def run_condition(name, interventions=None):
    """Run masked prediction with optional steering, return predicted codons."""
    if interventions:
        steered.set_interventions(interventions)
    else:
        steered.clear_interventions()
    
    preds = predict_masked(dna_seq, mask_positions)
    steered.clear_interventions()
    
    pred_codons = [preds[pos] for pos in mask_positions]
    cai = compute_cai(pred_codons)
    gc3 = compute_gc3(pred_codons)
    
    # Check amino acid preservation
    aa_preserved = sum(
        1 for pos in mask_positions
        if CODON_TO_AA.get(preds[pos], '?') == CODON_TO_AA.get(codons[pos], '!')
    )
    aa_rate = aa_preserved / len(mask_positions)
    
    print(f"{name}:")
    print(f"  CAI  = {cai:.4f}")
    print(f"  GC3  = {gc3:.4f}")
    print(f"  AA preserved: {aa_preserved}/{len(mask_positions)} ({aa_rate:.1%})")
    
    return {"name": name, "predictions": preds, "cai": cai, "gc3": gc3, "aa_rate": aa_rate}


# 1. Baseline
baseline = run_condition("Baseline")

# 2. Human-optimized: amplify wobble-GC features
gc_interventions = [
    Intervention(feature_id=fid, weight=STEER_WEIGHT, mode=InterventionMode.ADDITIVE_CODE)
    for fid in GC_STEER_FEATURES
]
optimized = run_condition("Human-optimized", gc_interventions)

# 3. De-optimized: amplify wobble-AT features
at_interventions = [
    Intervention(feature_id=fid, weight=STEER_WEIGHT, mode=InterventionMode.ADDITIVE_CODE)
    for fid in AT_STEER_FEATURES
]
deoptimized = run_condition("De-optimized", at_interventions)

## 9. Visualize Results

In [ ]:
# Bar chart: CAI and GC3 comparison
conditions = [baseline, optimized, deoptimized]
names = [c["name"] for c in conditions]
colors = ["#6baed6", "#2ca25f", "#e34a33"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# CAI
ax = axes[0]
cai_vals = [c["cai"] for c in conditions]
bars = ax.bar(names, cai_vals, color=colors, edgecolor="black", linewidth=0.5)
ax.set_ylabel("CAI")
ax.set_title("Codon Adaptation Index")
ax.axhline(compute_cai(codons), color="gray", linestyle="--", label="Original seq")
ax.legend(fontsize=9)
for bar, val in zip(bars, cai_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f"{val:.3f}",
            ha="center", va="bottom", fontsize=10)

# GC3
ax = axes[1]
gc3_vals = [c["gc3"] for c in conditions]
bars = ax.bar(names, gc3_vals, color=colors, edgecolor="black", linewidth=0.5)
ax.set_ylabel("GC3")
ax.set_title("GC Content at Wobble Position")
ax.axhline(compute_gc3(codons), color="gray", linestyle="--", label="Original seq")
ax.legend(fontsize=9)
for bar, val in zip(bars, gc3_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f"{val:.3f}",
            ha="center", va="bottom", fontsize=10)

# AA preservation
ax = axes[2]
aa_vals = [c["aa_rate"] for c in conditions]
bars = ax.bar(names, aa_vals, color=colors, edgecolor="black", linewidth=0.5)
ax.set_ylabel("AA Preservation Rate")
ax.set_title("Amino Acid Identity Preserved")
ax.set_ylim(0, 1.05)
for bar, val in zip(bars, aa_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{val:.1%}",
            ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

## 10. Sequence Alignment View

Show a per-position comparison of codon predictions across conditions.
Green = same amino acid as original; red = amino acid changed.

In [ ]:
def alignment_table(conditions, positions, codons_orig, n_show=30):
    """Print a text alignment of original vs predicted codons."""
    show_positions = positions[:n_show]
    
    print(f"{'Pos':>4}  {'Original':>8}  ", end="")
    for c in conditions:
        print(f"  {c['name']:>16}", end="")
    print()
    print("-" * (16 + 18 * len(conditions)))
    
    for pos in show_positions:
        orig = codons_orig[pos]
        orig_aa = CODON_TO_AA.get(orig, '?')
        print(f"{pos:>4}  {orig:>3} ({orig_aa})  ", end="")
        for c in conditions:
            pred = c["predictions"][pos]
            pred_aa = CODON_TO_AA.get(pred, '?')
            marker = " " if pred_aa == orig_aa else "*"
            syn = "syn" if pred != orig and pred_aa == orig_aa else ("   " if pred == orig else "MIS")
            print(f"  {pred:>3} ({pred_aa}){marker} {syn}", end="")
        print()


alignment_table(conditions, mask_positions, codons, n_show=40)

## 11. Codon Usage Distribution

Compare the distribution of predicted codons: do steered predictions shift toward
the expected codon usage patterns?

In [ ]:
def wobble_distribution(predictions, positions):
    """Count G/C vs A/T at wobble position for predicted codons."""
    gc, at = 0, 0
    for pos in positions:
        codon = predictions.get(pos, "???")
        if len(codon) == 3 and CODON_TO_AA.get(codon, '*') != '*':
            if codon[2] in ('G', 'C'):
                gc += 1
            else:
                at += 1
    return gc, at


fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(conditions))
width = 0.35

gc_counts = []
at_counts = []
for c in conditions:
    gc, at = wobble_distribution(c["predictions"], mask_positions)
    gc_counts.append(gc)
    at_counts.append(at)

bars1 = ax.bar(x - width/2, gc_counts, width, label="Wobble G/C", color="#2ca25f", edgecolor="black", linewidth=0.5)
bars2 = ax.bar(x + width/2, at_counts, width, label="Wobble A/T", color="#e34a33", edgecolor="black", linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylabel("Count")
ax.set_title("Wobble Position Nucleotide Distribution")
ax.legend()
plt.tight_layout()
plt.show()

## 12. Steering Strength Sweep

How does the steering weight affect CAI and GC3? Sweep from negative (suppress)
to positive (amplify) weights on the wobble-GC features.

In [ ]:
sweep_weights = [-20, -10, -5, 0, 5, 10, 15, 20, 30]
sweep_results = []

for w in sweep_weights:
    if w == 0:
        steered.clear_interventions()
    else:
        interventions = [
            Intervention(feature_id=fid, weight=w, mode=InterventionMode.ADDITIVE_CODE)
            for fid in GC_STEER_FEATURES
        ]
        steered.set_interventions(interventions)
    
    preds = predict_masked(dna_seq, mask_positions)
    pred_codons = [preds[pos] for pos in mask_positions]
    
    sweep_results.append({
        "weight": w,
        "cai": compute_cai(pred_codons),
        "gc3": compute_gc3(pred_codons),
    })
    print(f"  weight={w:>4}: CAI={sweep_results[-1]['cai']:.4f}  GC3={sweep_results[-1]['gc3']:.4f}")

steered.clear_interventions()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

weights = [r["weight"] for r in sweep_results]

ax1.plot(weights, [r["cai"] for r in sweep_results], "o-", color="#2ca25f", linewidth=2)
ax1.axhline(compute_cai(codons), color="gray", linestyle="--", alpha=0.7, label="Original")
ax1.axvline(0, color="gray", linestyle=":", alpha=0.5)
ax1.set_xlabel("Steering Weight")
ax1.set_ylabel("CAI")
ax1.set_title("CAI vs Steering Weight (wobble-GC features)")
ax1.legend()

ax2.plot(weights, [r["gc3"] for r in sweep_results], "o-", color="#e34a33", linewidth=2)
ax2.axhline(compute_gc3(codons), color="gray", linestyle="--", alpha=0.7, label="Original")
ax2.axvline(0, color="gray", linestyle=":", alpha=0.5)
ax2.set_xlabel("Steering Weight")
ax2.set_ylabel("GC3")
ax2.set_title("GC3 vs Steering Weight (wobble-GC features)")
ax2.legend()

plt.tight_layout()
plt.show()

## 13. Bonus: Single Amino Acid Feature Steering

Some SAE features are enriched for specific amino acids. For example, feature 795
is labeled "L (62%)" -- it fires at Leucine positions. Steering this feature should
change *which* Leucine codon is predicted (CTG vs CTT vs CTC vs CTA vs TTA vs TTG)
without affecting non-Leucine positions.

In [ ]:
# Find Leucine positions in the sequence
leu_positions = [i for i, c in enumerate(codons) if CODON_TO_AA.get(c) == 'L']
leu_mask = [p for p in leu_positions if 1 <= p < n_codons - 1]  # exclude first/last
print(f"Leucine positions: {len(leu_mask)} codons")

# Find amino-acid-specific features from labels
aa_features = {}
for fid_str, label in feature_labels.items():
    for aa_code in ['L ', 'A ', 'G ', 'V ', 'P ', 'S ', 'E ', 'K ', 'D ', 'R ']:
        if label.startswith(aa_code) and '(' in label and '%)' in label:
            aa_letter = aa_code.strip()
            aa_features.setdefault(aa_letter, []).append(int(fid_str))

if 'L' in aa_features:
    LEU_FEATURE = aa_features['L'][0]  # pick the first one
    print(f"Using Leucine feature: {LEU_FEATURE} ('{feature_labels[str(LEU_FEATURE)]}')") 
else:
    LEU_FEATURE = 795  # fallback
    print(f"Using hardcoded Leucine feature: {LEU_FEATURE}")

In [ ]:
# Predict Leucine codons under different steering of the Leu feature
leu_codons_all = ['CTG', 'CTC', 'CTT', 'CTA', 'TTG', 'TTA']

print(f"Leucine codon distribution at masked Leu positions:\n")
leu_results = []
for w in [-15, 0, 15, 30]:
    if w != 0:
        steered.set_interventions([
            Intervention(feature_id=LEU_FEATURE, weight=w, mode=InterventionMode.ADDITIVE_CODE)
        ])
    else:
        steered.clear_interventions()
    
    preds = predict_masked(dna_seq, leu_mask)
    pred_codons = [preds[p] for p in leu_mask]
    counts = Counter(pred_codons)
    
    print(f"  weight={w:>4}: ", end="")
    for lc in leu_codons_all:
        print(f"{lc}={counts.get(lc, 0):>3} ", end="")
    # Also count non-Leu predictions
    non_leu = sum(1 for c in pred_codons if CODON_TO_AA.get(c) != 'L')
    print(f"  (non-Leu: {non_leu})")
    leu_results.append({"weight": w, "counts": dict(counts), "non_leu": non_leu})

steered.clear_interventions()

In [ ]:
# Stacked bar chart of Leucine codon distribution
fig, ax = plt.subplots(figsize=(8, 5))

leu_colors = ['#1b9e77', '#d95f02', '#7570b3', '#e7298a', '#66a61e', '#e6ab02']
x = np.arange(len(leu_results))
bottoms = np.zeros(len(leu_results))

for i, lc in enumerate(leu_codons_all):
    vals = [r["counts"].get(lc, 0) for r in leu_results]
    ax.bar(x, vals, bottom=bottoms, label=lc, color=leu_colors[i], edgecolor="black", linewidth=0.3)
    bottoms += vals

ax.set_xticks(x)
ax.set_xticklabels([f"w={r['weight']}" for r in leu_results])
ax.set_ylabel("Count")
ax.set_title(f"Leucine Codon Choice Under Feature {LEU_FEATURE} Steering")
ax.legend(title="Codon", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated that SAE features learned from CodonFM's residual stream
capture biologically meaningful properties of codon usage. By steering these features
at inference time, we can:

1. **Shift codon optimization** -- amplifying "wobble GC" features increases CAI and GC3,
   effectively performing codon optimization through the model's learned representations

2. **Preserve protein identity** -- steering changes synonymous codon choice while
   largely preserving the encoded amino acid, showing the model disentangles these levels

3. **Target specific amino acids** -- amino-acid-specific features allow fine-grained
   control over codon choice at particular residue types

4. **Continuously control the effect** -- the steering weight acts as a dial, with
   monotonic effects on measurable codon usage metrics